# I. Introduction

This notebook is purposed to deliver modeling process from feature engineering to model saving.

output : Embedding Classification Model

Created by : Ghozie Ikhsan Fairuz

# II. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
import pickle

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')

# III. Data Loading

In [3]:
# Read file csv
df= pd.read_csv('McDonald_s_Reviews.csv', encoding='latin1')
print(df.shape)

# Summary Information of dfset
df.info()
print('\n')

# Summary statistic deskriptive numerical columns
print('Describe Numeric:')
display(df.describe(include=[np.number]).T)
print('\n')

# Duplicated df
print(f'Number of Duplicated df: {df.duplicated().sum()}\n')

# Missing Values df
print(f'Number of Missing Values df: {df.isnull().sum()}\n')

# Displays the first 10 df and the last 10 df
print("The first 10 df:")
display(df.head(10))

print("The Last 10 df:")
display(df.tail(10))

(33396, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33396 entries, 0 to 33395
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   reviewer_id    33396 non-null  int64  
 1   store_name     33396 non-null  object 
 2   category       33396 non-null  object 
 3   store_address  33396 non-null  object 
 4   latitude       32736 non-null  float64
 5   longitude      32736 non-null  float64
 6   rating_count   33396 non-null  object 
 7   review_time    33396 non-null  object 
 8   review         33396 non-null  object 
 9   rating         33396 non-null  object 
dtypes: float64(2), int64(1), object(7)
memory usage: 2.5+ MB


Describe Numeric:


,count,mean,std,min,25%,50%,75%,max
reviewer_id,33396.0,16698.500000,9640.739131,1.000000,8349.750000,16698.500000,25047.250000,33396.00000
latitude,32736.0,34.442546,5.344116,25.790295,28.655350,33.931261,40.727401,44.98141
longitude,32736.0,-90.647033,16.594844,-121.995421,-97.792874,-81.471414,-75.399919,-73.45982




Number of Duplicated df: 0

Number of Missing Values df: reviewer_id        0
store_name         0
category           0
store_address      0
latitude         660
longitude        660
rating_count       0
review_time        0
review             0
rating             0
dtype: int64

The first 10 df:


,reviewer_id,store_name,category,store_address,latitude,longitude,rating_count,review_time,review,rating
0,1,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",3 months ago,Why does it look like someone spit on my food?...,1 star
1,2,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",5 days ago,It'd McDonalds. It is what it is as far as the...,4 stars
2,3,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",5 days ago,Made a mobile order got to the speaker and che...,1 star
3,4,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",a month ago,My mc. Crispy chicken sandwich was ï¿½ï¿½ï¿½ï¿...,5 stars
4,5,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",2 months ago,"I repeat my order 3 times in the drive thru, a...",1 star
5,6,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",3 weeks ago,I work for door dash and they locked us all ou...,1 star
6,7,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",3 months ago,If I could give this location a zero on custo...,1 star
7,8,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",a year ago,Came in and ordered a Large coffee w/no ice. T...,1 star
8,9,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",3 months ago,Went thru drive thru. Ordered. Getting home no...,1 star
9,10,McDonald's,Fast food restaurant,"13749 US-183 Hwy, Austin, TX 78750, United States",30.460718,-97.792874,"1,240",3 months ago,"I'm not really a huge fan of fast food, but I ...",4 stars


The Last 10 df:


,reviewer_id,store_name,category,store_address,latitude,longitude,rating_count,review_time,review,rating
33386,33387,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",a year ago,Great in fast food ï¿½ï¿½ï¿,5 stars
33387,33388,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",6 years ago,"Excellent location and very good atmosphere, e...",5 stars
33388,33389,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",3 years ago,All very good food attention,5 stars
33389,33390,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",5 years ago,Quiet open the local January 1.,4 stars
33390,33391,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",a year ago,Very. Well,5 stars
33391,33392,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",4 years ago,They treated me very badly.,1 star
33392,33393,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",a year ago,The service is very good,5 stars
33393,33394,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",a year ago,To remove hunger is enough,4 stars
33394,33395,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",5 years ago,"It's good, but lately it has become very expen...",5 stars
33395,33396,McDonald's,Fast food restaurant,"3501 Biscayne Blvd, Miami, FL 33137, United St...",25.81,-80.189098,"2,810",2 years ago,they took good care of me,5 stars


# IV. Feature Engineering

## A. Text Normalization

### Add new column 'review_norm' by lowercasing, removing unique character and spaces

In [4]:
df['review_norm'] = (
    df['review']
    .str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df.review_norm.head()

0    why does it look like someone spit on my food ...
1    itd mcdonalds it is what it is as far as the f...
2    made a mobile order got to the speaker and che...
3    my mc crispy chicken sandwich was customer ser...
4    i repeat my order 3 times in the drive thru an...
Name: review_norm, dtype: object

### removing star or stars on column rating

In [5]:
df['rating'] = (
    df['rating']
    .str.replace(r'\s*stars?', '', regex=True)
    .str.strip()
    .astype(int)
)

df.rating.head()

0    1
1    4
2    1
3    5
4    1
Name: rating, dtype: int32

### Add new column rating class

In [6]:
rating_class=[]
for rate in df['rating']:
    if rate == 1:
        rating_class.append('negative')
    elif rate >1 and rate <= 4:
        rating_class.append('neutral')
    else :
        rating_class.append('positive')
        
df['rating_class']=rating_class

df.rating_class.head()

0    negative
1     neutral
2    negative
3    positive
4    negative
Name: rating_class, dtype: object

## B. Train Test split

In [7]:
X = df['review_norm'].tolist()
y = df['rating_class']

In [8]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# V. Modeling

## A. Model Definition

In this project, I used Sentence Transformers because this model is capable of representing text in the form of vectors that capture the contextual meaning of sentences. Unlike traditional methods such as TF-IDF, which only count word frequency, Sentence Transformers understand the relationships between words in a sentence, allowing them to capture semantic meaning more accurately. This selection is based on several key considerations:
- Able to capture the meaning and context of sentences, not just the frequency of words.
- Generating dense embeddings that represent semantic similarity between texts
- Effective for small to medium datasets without the need for fine-tuning large models
- More robust in handling variations of sentences with similar meanings

After the text is converted into numerical embeddings, I use Support Vector Machine (SVM) as the classification algorithm. SVM was chosen because it has strong performance on high-dimensional data, such as embeddings from transformers. Additionally, SVM is known for its stability and good generalization capabilities. The reasons for its selection include:
- Very effective for high-dimensional data (hundreds of embedding features)
- Able to form an optimal decision boundary with a maximum margin
- Reducing the risk of overfitting compared to complex models on limited datasets
- Fast and efficient for the training and inference process
-
Overall, the combination of Sentence Transformers and SVM provides a balance between deep semantic understanding and stable classification performance, making it suitable for sentiment analysis and text classification needs in this project.

## B. Model Embedding

In [17]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')


X_train_embed = embedder.encode(X_train_text, show_progress_bar=True)
X_test_embed = embedder.encode(X_test_text, show_progress_bar=True)

Batches:   0%|          | 0/835 [00:00<?, ?it/s]

Batches:   0%|          | 0/209 [00:00<?, ?it/s]

## C. Model Training

In [18]:
svc = LinearSVC(random_state=42)

model_svc=svc.fit(X_train_embed, y_train)

## D. Model Evaluation

In [19]:
y_pred_baseline = model_svc.predict(X_test_embed)
print(classification_report(y_test, y_pred_baseline))

              precision    recall  f1-score   support

    negative       0.74      0.80      0.77      1886
     neutral       0.69      0.66      0.67      2739
    positive       0.74      0.73      0.73      2055

    accuracy                           0.72      6680
   macro avg       0.72      0.73      0.72      6680
weighted avg       0.72      0.72      0.72      6680



After the model training, the results are quite good with an f1 score of around 67%-77% in all three categories. However, hyperparameter tuning will be attempted to achieve a comparison of more optimal model results.

# VI. Modeling using Hyperparameter Tuning

## A. Model Training

Model training using grid search hyperparameter tuning

In [20]:
param_grid = {
    'C': [0.01, 0.1, 1, 5, 10, 20],
    'loss': ['hinge', 'squared_hinge']
}

search = GridSearchCV(
    LinearSVC(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    verbose=2,
    n_jobs=1
)

search.fit(X_train_embed, y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END .................................C=0.01, loss=hinge; total time=   0.8s
[CV] END .................................C=0.01, loss=hinge; total time=   0.7s
[CV] END .................................C=0.01, loss=hinge; total time=   0.7s
[CV] END .........................C=0.01, loss=squared_hinge; total time=   2.3s
[CV] END .........................C=0.01, loss=squared_hinge; total time=   2.3s
[CV] END .........................C=0.01, loss=squared_hinge; total time=   2.4s
[CV] END ..................................C=0.1, loss=hinge; total time=   1.8s
[CV] END ..................................C=0.1, loss=hinge; total time=   1.7s
[CV] END ..................................C=0.1, loss=hinge; total time=   1.4s
[CV] END ..........................C=0.1, loss=squared_hinge; total time=   4.4s
[CV] END ..........................C=0.1, loss=squared_hinge; total time=   4.5s
[CV] END ..........................C=0.1, loss=s

GridSearchCV(cv=3, estimator=LinearSVC(random_state=42), n_jobs=1,
             param_grid={'C': [0.01, 0.1, 1, 5, 10, 20],
                         'loss': ['hinge', 'squared_hinge']},
             scoring='f1_macro', verbose=2)

## B. Model Evaluation

In [22]:
best_model = search.best_estimator_

y_pred_tuned = best_model.predict(X_test_embed)

print("Best Params:", search.best_params_)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_tuned))

Best Params: {'C': 1, 'loss': 'hinge'}

Classification Report:

              precision    recall  f1-score   support

    negative       0.73      0.82      0.77      1886
     neutral       0.70      0.63      0.67      2739
    positive       0.73      0.75      0.74      2055

    accuracy                           0.72      6680
   macro avg       0.72      0.73      0.73      6680
weighted avg       0.72      0.72      0.72      6680



# VII. Model Evaluation before and after tuning

In [23]:
before_tuning=classification_report(y_test, y_pred_baseline)
after_tuning=classification_report(y_test, y_pred_tuned)

print('Evaluation test set before tuning :',before_tuning)
print('Evaluation test set after tuning :',after_tuning),

Evaluation test set before tuning :               precision    recall  f1-score   support

    negative       0.74      0.80      0.77      1886
     neutral       0.69      0.66      0.67      2739
    positive       0.74      0.73      0.73      2055

    accuracy                           0.72      6680
   macro avg       0.72      0.73      0.72      6680
weighted avg       0.72      0.72      0.72      6680

Evaluation test set after tuning :               precision    recall  f1-score   support

    negative       0.73      0.82      0.77      1886
     neutral       0.70      0.63      0.67      2739
    positive       0.73      0.75      0.74      2055

    accuracy                           0.72      6680
   macro avg       0.72      0.73      0.73      6680
weighted avg       0.72      0.72      0.72      6680



(None,)

# VIII. Model Saving

In [24]:
with open('embedding_classification.pkl', 'wb') as model_file:
    pickle.dump(svc, model_file)

# IX. Conclusion

In general, the model's performance before and after tuning does not show significant improvement. Accuracy remains at 0.72, and both the macro average and weighted average values are also relatively stable. This means that the tuning process did not provide any substantial improvement to the overall model performance.

However, if we look at it in more detail per class:

- Negative Class

Recall increased (0.80 → 0.82)

Precision has slightly decreased (0.74 → 0.73)

F1-score remains (0.77)

That means, after tuning, the model is more aggressive in detecting the negative class (capturing more negative data), but with the consequence of a slight increase in false positives.

- Neutral Class

Precision increased (0.69 → 0.70)

Recall decreased (0.66 → 0.63)

F1-score remains the same (0.67)

Here, the model becomes more selective in predicting neutral, resulting in fewer false positives, but more neutrals go undetected (false negatives increase).

- Positive Class

Recall increased (0.73 → 0.75)

Precision slightly decreased (0.74 → 0.73)

F1-score slightly increased (0.73 → 0.74)

The model captures more positive data, but it comes with the consequence of increased false positives.


Next is why I chose to use the pre-tuning model because the difference between precision and recall is more balanced compared to the post-tuning model.

Why is this important?

Precision is directly related to false positives Low precision → high false positives

Recall is related to false negatives. Low recall → many data points that should have been detected but were missed.

In the model after tuning, there is a tendency for recall to increase in several classes but precision to slightly decrease. This indicates that the model is becoming more aggressive in predicting a class, which could potentially increase false positives.

By choosing the model before tuning:

The difference between precision and recall is smaller.

The model is more balanced in handling false positives and false negatives.

The risk of bias against certain classes is lower.

With the conclusion that although tuning improves recall in several classes, the overall improvement is not significant and is accompanied by a decrease in precision, indicating a potential increase in false positives.

Because the F1-score and accuracy are relatively stagnant, and the pre-tuning model has a more stable precision-recall balance, the pre-tuning model was chosen to maintain more consistent performance and minimize potential classification bias.